In [1]:
# ============================================================
# HKUST Hall Info Crawler → RAG-Optimized Markdown Generator
# ============================================================
# This notebook crawls HKUST SHRLO pages and directly produces
# well-structured Markdown files optimized for Alibaba Cloud
# Bailian knowledge base chunking (heading-based splitting).
#
# Output files (all in the current directory):
#   - hall_charges.md          (## = student type, with full table)
#   - hall_room_types.md       (## per hall, room types + mattress)
#   - hall_members.md          (## per hall, staff + tutors)
#   - hall_key_dates.md        (## per residential year)
#   - hall_admission_policy.md (## per policy part, ### per section)
#   - hall_facilities.md       (## per hall, ### per facility)
# ============================================================

import os
import requests
from bs4 import BeautifulSoup

# NOTE: The notebook lives inside outputs_md_crawler/, so "." is the output dir.
OUTPUT_DIR = "."
os.makedirs(OUTPUT_DIR, exist_ok=True)

BASE_URL = "https://shrl.hkust.edu.hk"

# --- Hall metadata for consistent naming across all files ---
HALL_META = [
    ("UG Hall I",    "ughall1"),
    ("UG Hall II",   "ughall2"),
    ("UG Hall III",  "ughall3"),
    ("UG Hall IV",   "ughall4"),
    ("UG Hall V",    "ughall5"),
    ("UG Hall VI",   "ughall6"),
    ("UG Hall VII",  "ughall7"),
    ("UG Hall VIII", "ughall8"),
    ("UG Hall IX",   "ughall9"),
    ("Jockey Club Hall (JCH)", "ughall-jch"),
]

# --- Helper functions ---
def fetch_soup(url):
    """Fetch a URL and return a BeautifulSoup object."""
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    return BeautifulSoup(resp.content, "html.parser")

def write_md(filename, content):
    """Write content to a markdown file under OUTPUT_DIR."""
    path = os.path.join(OUTPUT_DIR, filename)
    with open(path, "w", encoding="utf-8") as f:
        f.write(content)
    size_kb = len(content.encode("utf-8")) / 1024
    print(f"✅ {filename}  ({size_kb:.1f} KB, {content.count(chr(10))+1} lines)")

def clean_cell(text):
    """Clean a table cell value."""
    text = text.strip().replace("\xa0", " ")
    return text if text and text != "—" else "N/A"

print("✅ Imports and helpers ready.")

✅ Imports and helpers ready.


# HKUST Hall Info Crawler → RAG-Optimized Markdown

This notebook crawls HKUST Student Housing (SHRLO) pages and **directly produces structured Markdown files** optimized for Alibaba Cloud Bailian knowledge base chunking.

### Output Files
| File | Content | Chunk Level | Approx Chunks |
|------|---------|-------------|---------------|
| `hall_charges.md` | Hall fees by 4 student types | `##` | 4 |
| `hall_room_types.md` | Room types & mattress per hall | `##` | ~15 |
| `hall_members.md` | Staff & tutors per hall | `##` | 10 |
| `hall_key_dates.md` | Key dates by residential year | `##` | 2-3 |
| `hall_admission_policy.md` | Priority, Points, Lottery, Waitlist | `###` | ~15 |
| `hall_facilities.md` | Facility details per hall | `###` | ~60 |

In [13]:
# ============================================================
# 1. HALL CHARGES → hall_charges.md
# ============================================================
# Structure: # Title → ## per student type (with full pricing table)
# Chunk level: ## (4 chunks, each ~300 tokens with a complete table)
# ============================================================

CHARGE_URLS = {
    "New Local Students":          "https://shrl.hkust.edu.hk/apply-for-housing/ug/hall-charges-new-local",
    "New Non-Local Students":      "https://shrl.hkust.edu.hk/apply-for-housing/ug/hall-charges-new-non-local",
    "Continuing Local Students":   "https://shrl.hkust.edu.hk/apply-for-housing/ug/hall-charges-continuing-local",
    "Continuing Non-Local Students": "https://shrl.hkust.edu.hk/apply-for-housing/ug/hall-charges-continuing-non-local",
}

CHARGE_LABELS_CN = {
    "New Local Students":            "新入学本地学生",
    "New Non-Local Students":        "新入学非本地学生",
    "Continuing Local Students":     "在读本地学生",
    "Continuing Non-Local Students": "在读非本地学生",
}

lines = [
    "# HKUST Undergraduate Hall Charges (宿舍收费标准)",
    "",
    "本文档汇总了香港科技大学所有本科生宿舍的住宿费用，按四种学生类型分类。",
    "费用以港币（HK$）计算，按学年收取。",
    "",
]

for label, url in CHARGE_URLS.items():
    cn = CHARGE_LABELS_CN[label]
    print(f"  Crawling charges: {label} ...")
    soup = fetch_soup(url)
    table = soup.find("table", class_="tbl-border")
    rows = table.find_all("tr")

    # Parse header row
    headers = [clean_cell(c.text) for c in rows[0].find_all(["td", "th"])]

    lines.append("---")
    lines.append("")
    lines.append(f"## {label} - Hall Charges ({cn}宿舍收费)")
    lines.append("")
    lines.append(f"以下为**{cn}**各宿舍的住宿费用，单位为港币 HK$，按学年收取：")
    lines.append("")

    # Markdown table
    lines.append("| " + " | ".join(headers) + " |")
    lines.append("| " + " | ".join(["---"] * len(headers)) + " |")
    for row in rows[1:]:
        vals = [clean_cell(c.text) for c in row.find_all(["td", "th"])]
        # Pad if fewer columns
        while len(vals) < len(headers):
            vals.append("N/A")
        lines.append("| " + " | ".join(vals) + " |")
    lines.append("")

write_md("hall_charges.md", "\n".join(lines))
print(f"  → {len(CHARGE_URLS)} student type sections generated.")

  Crawling charges: New Local Students ...
  Crawling charges: New Non-Local Students ...
  Crawling charges: Continuing Local Students ...
  Crawling charges: Continuing Non-Local Students ...
✅ hall_charges.md  (3.3 KB, 97 lines)
  → 4 student type sections generated.


In [14]:
# ============================================================
# 2. ROOM TYPES → hall_room_types.md
# ============================================================
# Structure: # Title → ## per hall (room counts + mattress size)
# Chunk level: ## (15 chunks, one per hall, each self-contained)
# ============================================================

ROOM_URL = "https://shrl.hkust.edu.hk/apply-for-housing/ug/number-of-hall-places-and-room-types"

print("  Crawling room types ...")
soup = fetch_soup(ROOM_URL)
table = soup.find("table", class_="tbl-border")
rows = table.find_all("tr")

# Parse header
headers = [clean_cell(c.text) for c in rows[0].find_all(["td", "th"])]

# Parse data rows with merge handling (Hall V has a continuation row)
parsed = []
for row in rows[1:]:
    vals = [clean_cell(c.text) for c in row.find_all(["td", "th"])]
    # Detect continuation row: first cell starts with "(" like "(Bottom)..."
    if vals and vals[0].startswith("("):
        if parsed:
            # Merge into the last column (Mattress) of the previous row
            parsed[-1][-1] += " / " + vals[0]
        continue
    # Pad if fewer columns
    while len(vals) < len(headers):
        vals.append("N/A")
    parsed.append(vals)

# Map Roman numerals to full names for headings
ROMAN_TO_NAME = {}
for name, code in HALL_META:
    roman = name.replace("UG Hall ", "").replace("Jockey Club Hall (JCH)", "JCH")
    # Also handle JCH^ with footnote marker
    ROMAN_TO_NAME[roman] = (name, code)
    ROMAN_TO_NAME[roman + "^"] = (name, code)

# Room type column indices (skip first col = hall name, last col = mattress)
room_type_cols = headers[1:-1]  # e.g., Double Room, Triple Room, Bunk Bed Room, Single Room
mattress_col_idx = len(headers) - 1

lines = [
    "# HKUST Undergraduate Hall Room Types and Capacity (宿舍房型与床位数量)",
    "",
    "本文档汇总了香港科技大学所有本科生宿舍的房间类型、可容纳人数及床垫尺寸。",
    "",
]

for row_data in parsed:
    hall_key = row_data[0].strip()
    lookup = ROMAN_TO_NAME.get(hall_key)
    if lookup:
        hall_display, hall_code = lookup
        heading = f"## {hall_display} ({hall_code}) - Room Types & Capacity"
    else:
        # Halls X-XIII that are not in HALL_META
        heading = f"## UG Hall {hall_key} - Room Types & Capacity"

    lines.append("---")
    lines.append("")
    lines.append(heading)
    lines.append("")

    # List available room types with counts
    for i, rt in enumerate(room_type_cols, start=1):
        count = row_data[i] if i < len(row_data) else "N/A"
        if count != "N/A":
            lines.append(f"- **{rt}**: {count} places")

    # Mattress size
    mattress = row_data[mattress_col_idx] if mattress_col_idx < len(row_data) else "N/A"
    if mattress != "N/A":
        lines.append(f"- **Mattress Size**: {mattress}")

    lines.append("")

write_md("hall_room_types.md", "\n".join(lines))
print(f"  → {len(parsed)} hall sections generated.")

  Crawling room types ...
✅ hall_room_types.md  (2.3 KB, 113 lines)
  → 14 hall sections generated.


In [12]:
# ============================================================
# 3. HALL MEMBERS (Staff & Tutors) → hall_members.md
# ============================================================
# Structure: # Title → ## per hall (residence master, RLO, tutor table)
# Chunk level: ## (10 chunks, one per hall, each self-contained)
#
# Uses the proven original parsing logic (profile-image + gray-line-table).
# ============================================================

MEMBER_URLS = [
    ("UG Hall I",    "ughall1", "https://shrl.hkust.edu.hk/residential-halls/ug/ughall1"),
    ("UG Hall II",   "ughall2", "https://shrl.hkust.edu.hk/residential-halls/ug/ughall2"),
    ("UG Hall III",  "ughall3", "https://shrl.hkust.edu.hk/residential-halls/ug/ughall3"),
    ("UG Hall IV",   "ughall4", "https://shrl.hkust.edu.hk/residential-halls/ug/ughall4"),
    ("UG Hall V",    "ughall5", "https://shrl.hkust.edu.hk/residential-halls/ug/ughall5"),
    ("UG Hall VI",   "ughall6", "https://shrl.hkust.edu.hk/residential-halls/ug/ughall6"),
    ("UG Hall VII",  "ughall7", "https://shrl.hkust.edu.hk/residential-halls/ug/ughall7"),
    ("UG Hall VIII", "ughall8", "https://shrl.hkust.edu.hk/residential-halls/ug/ughall8"),
    ("UG Hall IX",   "ughall9", "https://shrl.hkust.edu.hk/residential-halls/ug/ughall9"),
    ("Jockey Club Hall (JCH)", "ughall-jch", "https://shrl.hkust.edu.hk/residential-halls/ug/ughall-jch"),
]


def crawl_and_format_members(soup, hall_name, hall_code):
    """Parse a hall page and return MD lines for the member section.
    
    Parsing strategy:
    1. profile-image table → RM (Residence Master) info via multi-line text
    2. gray-line-table team → first entry without floor/room = RLO, rest = tutors
    """
    section_lines = []
    
    # --- Extract hall inquiry email ---
    inquiry_email = ""
    for a in soup.find_all("a", href=True):
        if a["href"].startswith("mailto:") and "ust.hk" in a["href"]:
            candidate = a["href"].replace("mailto:", "").strip()
            if candidate.startswith("ug") or candidate.startswith("jch"):
                inquiry_email = candidate
                break
    
    section_lines.append(f"## {hall_name} ({hall_code}) - Staff & Tutors")
    section_lines.append("")
    if inquiry_email:
        section_lines.append(f"- **Hall Inquiry Email**: {inquiry_email}")
        section_lines.append("")
    
    # === Staff from profile-image table ===
    table = soup.find("table", class_="profile-image")
    if table:
        for row in table.find_all("tr"):
            cells = row.find_all(["td", "th"])
            cols = [ele.text.strip() for ele in cells]
            cols = [ele.replace("Tel: ", "").replace("Email: ", "") for ele in cols]
            cols = [ele for ele in cols if ele]
            if not cols:
                continue
            parts = cols[0].split("\n")
            parts = [p.strip() for p in parts if p.strip()]
            if len(parts) >= 2:
                name = parts[0]
                position = parts[1]
                tel = parts[2] if len(parts) > 2 else ""
                email = parts[3] if len(parts) > 3 else ""
                section_lines.append(f"**{position}**: {name}  ")
                if tel:
                    section_lines.append(f"- Tel: {tel}")
                if email:
                    section_lines.append(f"- Email: {email}")
                section_lines.append("")
    
    # === Entries from gray-line-table team ===
    all_entries = []
    for tbl in soup.find_all("table", class_="gray-line-table team"):
        thead = tbl.find("thead")
        headers = []
        if thead:
            headers = [th.text.strip() for th in thead.find_all("th")]
        
        for row in tbl.find_all("tr"):
            tds = row.find_all("td")
            if not tds:
                continue
            vals = [td.text.strip() for td in tds]
            if not any(vals):
                continue
            
            entry = {"name": "", "email": "", "floor": "", "room": ""}
            for i, v in enumerate(vals):
                if i < len(headers):
                    h = headers[i].lower()
                    if "name" in h:
                        entry["name"] = v
                    elif "email" in h or "mail" in h:
                        entry["email"] = v
                    elif h == "tutor" or "floor" in h:
                        entry["floor"] = v
                    elif "room" in h:
                        entry["room"] = v
                    else:
                        if i == 0 and not entry["name"]:
                            entry["name"] = v
                        elif not entry["email"] and "@" in v:
                            entry["email"] = v
                elif v:
                    if not entry["floor"]:
                        entry["floor"] = v
                    elif not entry["room"]:
                        entry["room"] = v
            
            # Smart fallback: detect email by @ symbol
            for i, v in enumerate(vals):
                if "@" in v and not entry["email"]:
                    entry["email"] = v
            
            if entry["name"]:
                all_entries.append(entry)
    
    # --- Separate RLO (no floor/room) from actual tutors ---
    rlo_entries = [e for e in all_entries if not e["floor"] and not e["room"]]
    tutor_entries = [e for e in all_entries if e["floor"] or e["room"]]
    
    # Format RLO as "Residential Life Officer"
    for rlo in rlo_entries:
        section_lines.append(f"**Residential Life Officer**: {rlo['name']}  ")
        if rlo["email"]:
            section_lines.append(f"- Email: {rlo['email']}")
        section_lines.append("")
    
    # Format tutors as a table
    if tutor_entries:
        section_lines.append("**Hall Tutors**:")
        section_lines.append("")
        section_lines.append("| Name | Floor | Room | Email |")
        section_lines.append("|------|-------|------|-------|")
        for t in tutor_entries:
            section_lines.append(
                f"| {t['name']} | {t['floor']} | {t['room']} | {t['email']} |"
            )
        section_lines.append("")
    
    return section_lines


# --- Build the consolidated MD ---
lines = [
    "# HKUST Undergraduate Hall Staff and Tutors (宿舍管理人员及导师)",
    "",
    "本文档汇总了香港科技大学所有本科生宿舍的管理团队信息，包括舍监（Residence Master）、",
    "宿舍生活主任（Residential Life Officer）及楼层导师（Hall Tutor）的联系方式。",
    "",
]

for hall_name, hall_code, url in MEMBER_URLS:
    print(f"  Crawling members: {hall_name} ...")
    soup = fetch_soup(url)
    lines.append("---")
    lines.append("")
    section = crawl_and_format_members(soup, hall_name, hall_code)
    lines.extend(section)

write_md("hall_members.md", "\n".join(lines))
print(f"  → {len(MEMBER_URLS)} hall sections generated.")

  Crawling members: UG Hall I ...
  Crawling members: UG Hall II ...
  Crawling members: UG Hall III ...
  Crawling members: UG Hall IV ...
  Crawling members: UG Hall V ...
  Crawling members: UG Hall VI ...
  Crawling members: UG Hall VII ...
  Crawling members: UG Hall VIII ...
  Crawling members: UG Hall IX ...
  Crawling members: Jockey Club Hall (JCH) ...
✅ hall_members.md  (7.3 KB, 250 lines)
  → 10 hall sections generated.


In [15]:
# ============================================================
# 4. KEY DATES → hall_key_dates.md
# ============================================================
# Structure: # Title → ## per residential year group
# Chunk level: ## (2-3 chunks, grouped by RY)
# ============================================================

KEY_DATES_URL = "https://shrl.hkust.edu.hk/apply-for-housing/ug/new-local-ugs"

print("  Crawling key dates ...")
soup = fetch_soup(KEY_DATES_URL)

# Find all calendar-details tables inside mtpc-2col-item--2 sections
items = soup.find_all("div", class_="mtpc-2col-item mtpc-2col-item--2")
events = []
for item in items:
    table = item.find("table", class_="calendar-details")
    if table:
        for row in table.find_all("tr"):
            cols = row.find_all(["td", "th"])
            if len(cols) >= 2:
                date = cols[0].get_text(strip=True).replace("\xa0", " ")
                event = cols[1].get_text(strip=True).replace("\xa0", " ")
                if date and event:
                    events.append((date, event))

# Group events by Residential Year
from collections import OrderedDict
ry_groups = OrderedDict()
for date, event in events:
    # Extract RY info from event text (e.g., "RY2025-26" or "RY2026-27")
    import re
    ry_match = re.search(r"RY(\d{4}-\d{2,4})", event)
    if ry_match:
        ry = ry_match.group(1)
        # Normalize: "2025-26" format
        if len(ry.split("-")[1]) == 4:
            ry = ry[:4] + "-" + ry[-2:]
    else:
        ry = "General"
    ry_groups.setdefault(ry, []).append((date, event))

lines = [
    "# HKUST UG Hall Key Dates and Deadlines (宿舍重要日期)",
    "",
    "本文档汇总了香港科技大学本科生宿舍申请、缴费、入住和退宿的关键日期。",
    "",
]

for ry, ry_events in ry_groups.items():
    lines.append("---")
    lines.append("")
    if ry == "General":
        lines.append("## General Key Dates (其他重要日期)")
    else:
        lines.append(f"## Residential Year {ry} Key Dates ({ry}学年重要日期)")
    lines.append("")
    lines.append("| Date | Event |")
    lines.append("|------|-------|")
    for date, event in ry_events:
        # Clean up event text for table cell
        event_clean = event.replace("|", "/")
        lines.append(f"| {date} | {event_clean} |")
    lines.append("")

write_md("hall_key_dates.md", "\n".join(lines))
print(f"  → {len(ry_groups)} residential year sections, {len(events)} events total.")

  Crawling key dates ...
✅ hall_key_dates.md  (1.4 KB, 34 lines)
  → 2 residential year sections, 16 events total.


In [16]:
# ============================================================
# 5. ADMISSION POLICY → hall_admission_policy.md
# ============================================================
# Structure: # Title → ## per part → ### per sub-section
# Chunk level: ### (semantically complete policy sections)
#
# Also downloads PDF annexes referenced in the policy pages.
# ============================================================

POLICY_URLS = [
    ("priority-housing",     "https://shrl.hkust.edu.hk/admission-policy/ug/priority-housing"),
    ("hall-point-system-i",  "https://shrl.hkust.edu.hk/admission-policy/ug/hall-point-system-i"),
    ("lottery",              "https://shrl.hkust.edu.hk/admission-policy/ug/lottery"),
    ("waitlist",             "https://shrl.hkust.edu.hk/admission-policy/ug/waitlist"),
]

# --- Download PDF annexes ---
def download_pdfs(soup, save_dir):
    """Find and download PDF links from a page."""
    downloaded = []
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if href.endswith(".pdf"):
            filename = href.split("/")[-1]
            filepath = os.path.join(save_dir, filename)
            if not os.path.exists(filepath):
                full_url = href if href.startswith("http") else BASE_URL + href
                try:
                    r = requests.get(full_url, timeout=30)
                    with open(filepath, "wb") as f:
                        f.write(r.content)
                    downloaded.append(filename)
                except Exception as e:
                    print(f"    ⚠️ Failed to download {filename}: {e}")
    return downloaded


# --- Parse policy content ---
def parse_policy_content(soup):
    """Extract policy text and tables from the main content area."""
    content_div = soup.find("div", class_="mtpc-2col-item mtpc-2col-item--1")
    if not content_div:
        return []

    elements = []
    for el in content_div.find_all(["p", "table"]):
        if el.name == "p":
            text = el.get_text(" ", strip=True).replace("\xa0", " ")
            if text and text != "Back":
                elements.append(("text", text))
        elif el.name == "table":
            rows = []
            for tr in el.find_all("tr"):
                cells = tr.find_all(["td", "th"])
                row = [c.get_text(" ", strip=True).replace("\xa0", " ") for c in cells]
                if any(row):
                    rows.append(row)
            if rows:
                elements.append(("table", rows))
    return elements


# === Build the MD ===
pdf_dir = os.path.join(OUTPUT_DIR, "policy_pdfs")
os.makedirs(pdf_dir, exist_ok=True)

lines = [
    "# HKUST Undergraduate Hall Admission Policy (宿舍入住政策)",
    "",
    "本文档汇总了香港科技大学本科生宿舍的入住分配政策，包括优先住宿（Priority Housing）、",
    "积分制度（Hall Point System）、抽签（Lottery）和等候名单（Waitlist）四个部分。",
    "",
]

# ---- Part 1: Priority Housing ----
print("  Crawling policy: Priority Housing ...")
soup = fetch_soup(POLICY_URLS[0][1])
pdfs = download_pdfs(soup, pdf_dir)
if pdfs:
    print(f"    Downloaded PDFs: {pdfs}")

elements = parse_policy_content(soup)

lines.append("---")
lines.append("")
lines.append("## Part 1 - Priority Housing (优先住宿)")
lines.append("")
lines.append("Hall places are first allocated through priority housing to eligible student groups as follows.")
lines.append("")

# Parse the sections (1.1 through 1.8) from the content
current_section = None
section_buffer = []

SECTION_TITLES = {
    "1.1": "Non-Local Students (非本地学生)",
    "1.2": "First Year Local Students (一年级本地学生)",
    "1.3": "Local Students with No-Home-Base (无家庭基地本地学生)",
    "1.4": "Local Students with TT over 120 Minutes (通勤超120分钟本地学生)",
    "1.5": "Exchange-In Students (交换生)",
    "1.6": "University-approved Schemes or Programs (大学项目学生)",
    "1.7": "Decision by Hall Admissions Committee (入住委员会决定)",
    "1.8": "Procedures Reference (详细流程参考)",
}

def flush_section(lines, section_id, buffer):
    """Flush accumulated content for a section."""
    if section_id and buffer:
        title = SECTION_TITLES.get(section_id, section_id)
        lines.append(f"### Priority Housing - Section {section_id}: {title}")
        lines.append("")
        for item_type, item_data in buffer:
            if item_type == "text":
                lines.append(item_data)
                lines.append("")
            elif item_type == "table":
                # Format as markdown table
                max_cols = max(len(r) for r in item_data)
                lines.append("| " + " | ".join([f"Col {j+1}" for j in range(max_cols)]) + " |")
                lines.append("| " + " | ".join(["---"] * max_cols) + " |")
                for row in item_data:
                    while len(row) < max_cols:
                        row.append("")
                    lines.append("| " + " | ".join(r.replace("|", "/") for r in row) + " |")
                lines.append("")

for el_type, el_data in elements:
    if el_type == "text":
        # Check if this starts a new section (e.g., "1.1 Non-Local students...")
        import re
        sec_match = re.match(r"^(1\.\d)\s", el_data)
        if sec_match:
            # Flush previous section
            flush_section(lines, current_section, section_buffer)
            current_section = sec_match.group(1)
            section_buffer = [("text", el_data)]
        else:
            if current_section:
                section_buffer.append(("text", el_data))
            else:
                # Pre-section content
                lines.append(el_data)
                lines.append("")
    elif el_type == "table":
        if current_section:
            section_buffer.append(("table", el_data))

# Flush the last section
flush_section(lines, current_section, section_buffer)


# ---- Part 2a: Hall Point System ----
print("  Crawling policy: Hall Point System ...")
soup = fetch_soup(POLICY_URLS[1][1])
elements = parse_policy_content(soup)

lines.append("---")
lines.append("")
lines.append("## Part 2a - Hall Point System (积分制度)")
lines.append("")

# All content goes into one ### chunk
lines.append("### Hall Point System - Criteria and Point Allocation (积分分配标准)")
lines.append("")

for el_type, el_data in elements:
    if el_type == "text":
        if el_data.strip() == "Back":
            continue
        lines.append(el_data)
        lines.append("")
    elif el_type == "table":
        # This is the points table
        lines.append("| Section | Allocation Criteria | Max Points (Local) | Max Points (Non-Local/NHB) |")
        lines.append("|---------|--------------------|--------------------|---------------------------|")
        for row in el_data:
            # Skip header rows (already defined above) and empty rows
            if not any(row) or (len(row) >= 2 and row[0] == "Section"):
                continue
            while len(row) < 4:
                row.append("")
            vals = [r.replace("|", "/") for r in row]
            lines.append("| " + " | ".join(vals) + " |")
        lines.append("")


# ---- Part 2b: Lottery ----
print("  Crawling policy: Lottery ...")
soup = fetch_soup(POLICY_URLS[2][1])
elements = parse_policy_content(soup)

lines.append("---")
lines.append("")
lines.append("## Part 2b - Lottery (抽签)")
lines.append("")
lines.append("### Lottery - Non-Local/NHB Random Draw (非本地/NHB随机抽签)")
lines.append("")

for el_type, el_data in elements:
    if el_type == "text" and el_data.strip() != "Back":
        lines.append(el_data)
        lines.append("")


# ---- Part 3: Waitlist ----
print("  Crawling policy: Waitlist ...")
soup = fetch_soup(POLICY_URLS[3][1])
elements = parse_policy_content(soup)

lines.append("---")
lines.append("")
lines.append("## Part 3 - Waitlist (等候名单)")
lines.append("")
lines.append("### Waitlist - Mechanism and Ranking (等候名单机制)")
lines.append("")

for el_type, el_data in elements:
    if el_type == "text" and el_data.strip() != "Back":
        lines.append(el_data)
        lines.append("")


write_md("hall_admission_policy.md", "\n".join(lines))
print(f"  → 4 policy parts generated.")

  Crawling policy: Priority Housing ...
    Downloaded PDFs: ['Annex_1-Guideline_for_Special_Application_with_Exceptional_Hardship.pdf', 'Annex_2-Guideline_for_Part_1-Priority_Housing.pdf', 'Example-Part_1.Priority_Housing.pdf']
  Crawling policy: Hall Point System ...
  Crawling policy: Lottery ...
  Crawling policy: Waitlist ...
✅ hall_admission_policy.md  (6.8 KB, 114 lines)
  → 4 policy parts generated.


In [17]:
# ============================================================
# 6. FACILITIES → hall_facilities.md
# ============================================================
# Structure: # Title → ## per hall → ### per facility category
# Chunk level: ### (self-contained, ~27-63 chunks)
# ============================================================

FACILITY_URLS = [
    ("UG Hall I",    "ughall1", "https://shrl.hkust.edu.hk/residential-halls/ug/ughall1"),
    ("UG Hall II",   "ughall2", "https://shrl.hkust.edu.hk/residential-halls/ug/ughall2"),
    ("UG Hall III",  "ughall3", "https://shrl.hkust.edu.hk/residential-halls/ug/ughall3"),
    ("UG Hall IV",   "ughall4", "https://shrl.hkust.edu.hk/residential-halls/ug/ughall4"),
    ("UG Hall V",    "ughall5", "https://shrl.hkust.edu.hk/residential-halls/ug/ughall5"),
    ("UG Hall VI",   "ughall6", "https://shrl.hkust.edu.hk/residential-halls/ug/ughall6"),
    ("UG Hall VII",  "ughall7", "https://shrl.hkust.edu.hk/residential-halls/ug/ughall7"),
    ("UG Hall VIII", "ughall8", "https://shrl.hkust.edu.hk/residential-halls/ug/ughall8"),
    ("UG Hall IX",   "ughall9", "https://shrl.hkust.edu.hk/residential-halls/ug/ughall9"),
    ("Jockey Club Hall (JCH)", "ughall-jch", "https://shrl.hkust.edu.hk/residential-halls/ug/ughall-jch"),
]


def parse_facilities(soup):
    """Parse facility categories and descriptions from a hall page.
    
    Returns: [(category_name, description_text), ...]
    """
    # Find the accordion section that lists facility categories
    # Categories are listed as <li> items, descriptions follow as section items
    categories = []
    descriptions = []
    
    # Strategy: find all section items and detect category list vs description text
    items = []
    for i in range(1, 30):
        item = soup.find("div", class_=f"field__item mtpc-section-item mtpc-section-item--{i}")
        if item:
            items.append((i, item))
    
    # Find the item that contains the facility category list (an <ul> with <li>)
    cat_item_idx = None
    for idx, (i, item) in enumerate(items):
        # Check if this item has an accordion-question (skip those - they are intro sections)
        question = item.find("div", class_="accordion-question")
        if question:
            continue
        # Check for a <ul> containing facility categories
        ul = item.find("ul")
        if ul and ul.find_all("li"):
            cat_item_idx = idx
            for li in ul.find_all("li"):
                categories.append(li.text.strip())
            break
    
    if cat_item_idx is None:
        return []
    
    # Descriptions follow the category list item
    start_idx = cat_item_idx + 1
    for j in range(start_idx, min(start_idx + len(categories), len(items))):
        _, item = items[j]
        text = item.get_text(" ", strip=True)
        text = text.replace("Text Area", "").strip()
        descriptions.append(text)
    
    # Pair categories with descriptions
    result = []
    for k in range(min(len(categories), len(descriptions))):
        result.append((categories[k], descriptions[k]))
    
    return result


# --- Build the consolidated MD ---
lines = [
    "# HKUST Undergraduate Hall Facilities (宿舍设施详情)",
    "",
    "本文档汇总了香港科技大学所有本科生宿舍的设施信息，按宿舍分类，",
    "涵盖卧室、公共休息室、茶水间、浴室、洗衣房等设施的详细描述。",
    "",
]

total_chunks = 0
for hall_name, hall_code, url in FACILITY_URLS:
    print(f"  Crawling facilities: {hall_name} ...")
    soup = fetch_soup(url)
    
    # Extract hall inquiry email
    inquiry_email = ""
    for a in soup.find_all("a", href=True):
        if a["href"].startswith("mailto:"):
            candidate = a["href"].replace("mailto:", "").strip()
            if "ust.hk" in candidate and (candidate.startswith("ug") or candidate.startswith("jch")):
                inquiry_email = candidate
                break
    
    facilities = parse_facilities(soup)
    
    lines.append("---")
    lines.append("")
    lines.append(f"## {hall_name} ({hall_code}) Facilities")
    lines.append("")
    if inquiry_email:
        lines.append(f"- **Hall Code**: {hall_code}")
        lines.append(f"- **Inquiry Email**: {inquiry_email}")
        lines.append("")
    
    if not facilities:
        lines.append("_(Facility data could not be parsed from this page.)_")
        lines.append("")
        continue
    
    for cat_name, cat_desc in facilities:
        lines.append(f"### {hall_name} ({hall_code}) - {cat_name}")
        lines.append("")
        lines.append(cat_desc)
        lines.append("")
        total_chunks += 1

write_md("hall_facilities.md", "\n".join(lines))
print(f"  → {len(FACILITY_URLS)} halls, {total_chunks} facility chunks generated.")

  Crawling facilities: UG Hall I ...
  Crawling facilities: UG Hall II ...
  Crawling facilities: UG Hall III ...
  Crawling facilities: UG Hall IV ...
  Crawling facilities: UG Hall V ...
  Crawling facilities: UG Hall VI ...
  Crawling facilities: UG Hall VII ...
  Crawling facilities: UG Hall VIII ...
  Crawling facilities: UG Hall IX ...
  Crawling facilities: Jockey Club Hall (JCH) ...
✅ hall_facilities.md  (14.9 KB, 255 lines)
  → 10 halls, 45 facility chunks generated.


In [18]:
# ============================================================
# 7. SUMMARY & VERIFICATION
# ============================================================
# List all generated RAG-optimized MD files with stats.
# ============================================================

import glob

md_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.md")))

print("=" * 60)
print(" Generated RAG-Optimized Markdown Files")
print("=" * 60)
print(f"{'File':<45} {'Size':>8}  {'## chunks':>10}  {'### chunks':>11}")
print("-" * 60)

total_h2 = 0
total_h3 = 0
for f in md_files:
    name = os.path.basename(f)
    size = os.path.getsize(f)
    with open(f, "r", encoding="utf-8") as fp:
        content = fp.read()
    # Count ## and ### headings
    import re
    h2_count = len(re.findall(r"^## ", content, re.MULTILINE))
    h3_count = len(re.findall(r"^### ", content, re.MULTILINE))
    total_h2 += h2_count
    total_h3 += h3_count
    size_str = f"{size/1024:.1f} KB"
    print(f"{name:<45} {size_str:>8}  {h2_count:>10}  {h3_count:>11}")

print("-" * 60)
print(f"{'TOTAL':<45} {'':>8}  {total_h2:>10}  {total_h3:>11}")
print()
print("📋 Bailian Upload Suggestions:")
print("  - hall_charges.md         → 按标题切分, ## level (二级标题), max 512")
print("  - hall_room_types.md      → 按标题切分, ## level (二级标题), max 512")
print("  - hall_members.md         → 按标题切分, ## level (二级标题), max 512")
print("  - hall_key_dates.md       → 按标题切分, ## level (二级标题), max 512")
print("  - hall_admission_policy.md→ 按标题切分, ### level (三级标题), max 512")
print("  - hall_facilities.md      → 按标题切分, ### level (三级标题), max 512")
print("  - 宿舍评价 files           → 按标题切分, ### level (三级标题), max 512")

 Generated RAG-Optimized Markdown Files
File                                              Size   ## chunks   ### chunks
------------------------------------------------------------
HKUST_Hall_Reviews_EN.md                       18.2 KB           9           27
hall_admission_policy.md                        6.8 KB           4           11
hall_charges.md                                 3.3 KB           4            0
hall_facilities.md                             14.9 KB          10           45
hall_key_dates.md                               1.4 KB           2            0
hall_members.md                                 7.3 KB          10            0
hall_room_types.md                              2.3 KB          14            0
香港科技大学宿舍评价（简介、地理位置、住宿体验）.md                    14.6 KB           9           27
香港科技大学宿舍评价（简介、地理位置、住宿体验）_backup.md             13.1 KB           0            0
------------------------------------------------------------
TOTAL                                 

In [ ]:
# ============================================================
# 8. ZIP all outputs for Bailian upload
# ============================================================

import zipfile
from pathlib import Path

def zip_folder(folder_path, zip_path=None):
    folder = Path(folder_path)
    if zip_path is None:
        zip_path = str(folder.with_suffix(".zip"))
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
        for file in folder.rglob("*"):
            if file.is_file() and not file.name.startswith("."):
                z.write(file, file.relative_to(folder))
    size_mb = os.path.getsize(zip_path) / (1024 * 1024)
    print(f"📦 Zipped to {zip_path} ({size_mb:.2f} MB)")
    return zip_path

zip_folder(OUTPUT_DIR)

✅ Converted 37 files to Markdown under /Users/wangjunyuan/桌面副本/fyp /outputs_md_crawler
 - data/hall_info/charges/local_new.csv -> outputs_md_crawler/charges/local_new.md
 - data/hall_info/charges/local_continuing.csv -> outputs_md_crawler/charges/local_continuing.md
 - data/hall_info/charges/non_local_continuing.csv -> outputs_md_crawler/charges/non_local_continuing.md
 - data/hall_info/charges/non_local_new.csv -> outputs_md_crawler/charges/non_local_new.md
 - data/hall_info/room_types/room_types.csv -> outputs_md_crawler/room_types/room_types.md
 - data/hall_info/member/ughall8_member.csv -> outputs_md_crawler/member/ughall8_member.md
 - data/hall_info/member/ughall5_member.csv -> outputs_md_crawler/member/ughall5_member.md
 - data/hall_info/member/ughall-jch_member.csv -> outputs_md_crawler/member/ughall-jch_member.md
 - data/hall_info/member/ughall2_member.csv -> outputs_md_crawler/member/ughall2_member.md
 - data/hall_info/member/ughall7_member.csv -> outputs_md_crawler/member/ugh

In [4]:
# ============================================================
# 9. SHRLO INFO → Hall data 3:3/shrlo info/
# ============================================================
# Crawls About SHRLO, Contact Us, and all Hall Life sub-pages.
# Produces categorized MD files optimized for Bailian RAG chunking.
#
# Output files (in Hall data 3:3/shrlo info/):
#   - shrlo_about_and_contact.md  (## About SHRLO + ## Contact Us)
#   - hall_life_guide.md          (## per topic: safety, tips, etc.)
#   - hall_life_guest_pass.md     (## scheme + ### per T&C section)
#   - hall_life_services.md       (## A/C + ## Laundry + ## GGT)
#   - hall_life_checkin_checkout.md(## Check-in + ## Check-out)
# ============================================================

import os, re
from collections import OrderedDict

SHRLO_OUTPUT_DIR = os.path.join("Hall data 3:3", "shrlo info")
os.makedirs(SHRLO_OUTPUT_DIR, exist_ok=True)

# Breadcrumb / nav noise patterns to skip
BREADCRUMB_RE = re.compile(
    r"^(Residential Halls.*?/|About\s*/|Hall Life\s*/|"
    r"Check-in/out Procedures$|Safety and Security$|Hall Activities$|"
    r"Living Tips$|Reflection Room$|Guest Pass Scheme$|"
    r"Hall A/C and Laundry Services?$|GGT SSC Smartmeter System$|"
    r"About SHRLO$|Contact Us$)", re.IGNORECASE)

def is_noise(text):
    """Return True if text is breadcrumb / page-title nav noise."""
    t = text.strip()
    if BREADCRUMB_RE.match(t):
        return True
    if t in ("Looking for Help?", "Back", "Left Column", "Text Area",
             "Right Column", "Container", " "):
        return True
    return False

def write_shrlo_md(filename, content):
    """Write content to a markdown file under SHRLO_OUTPUT_DIR.
    Post-processes to remove breadcrumb / nav noise lines."""
    # Remove breadcrumb / noise lines
    cleaned = []
    for line in content.split("\n"):
        stripped = line.strip()
        if stripped and is_noise(stripped):
            continue
        cleaned.append(line)
    content = "\n".join(cleaned)
    # Remove consecutive blank lines (collapse to max 2)
    content = re.sub(r"\n{3,}", "\n\n", content)
    path = os.path.join(SHRLO_OUTPUT_DIR, filename)
    with open(path, "w", encoding="utf-8") as f:
        f.write(content)
    size_kb = len(content.encode("utf-8")) / 1024
    print(f"  ✅ {filename}  ({size_kb:.1f} KB, {content.count(chr(10))+1} lines)")


def extract_text_blocks(soup, container_selector=None):
    """Extract text paragraphs and tables from main content area."""
    if container_selector:
        containers = soup.select(container_selector)
    else:
        # Default: left column text areas
        containers = soup.select(".mtpc-2col-item.mtpc-2col-item--1")
        if not containers:
            containers = [soup.find("main") or soup]

    blocks = []
    for container in containers:
        for el in container.find_all(["h2", "h3", "h4", "h5", "p", "ul", "ol", "table",
                                       "div"], recursive=True):
            # Skip nested divs that aren't text areas
            if el.name == "div":
                cls = " ".join(el.get("class", []))
                if "text-area" not in cls and "field__item" not in cls:
                    continue
                # Only process direct text in text-area divs if no child elements
                if el.find(["h2", "h3", "h4", "h5", "p", "ul", "ol", "table"]):
                    continue

            if el.name in ("h2", "h3", "h4", "h5"):
                text = el.get_text(" ", strip=True).replace("\xa0", " ")
                if text and text not in ("Back", ""):
                    level = int(el.name[1])
                    blocks.append(("heading", level, text))

            elif el.name == "p":
                text = el.get_text(" ", strip=True).replace("\xa0", " ")
                if text and text not in ("Back", "", " "):
                    blocks.append(("text", 0, text))

            elif el.name in ("ul", "ol"):
                # Skip if parent is already captured
                if el.parent and el.parent.name in ("li",):
                    continue
                items = []
                for li in el.find_all("li", recursive=False):
                    li_text = li.get_text(" ", strip=True).replace("\xa0", " ")
                    if li_text:
                        items.append(li_text)
                if items:
                    blocks.append(("list", 0, items))

            elif el.name == "table":
                rows = []
                for tr in el.find_all("tr"):
                    cells = [c.get_text(" ", strip=True).replace("\xa0", " ")
                             for c in tr.find_all(["td", "th"])]
                    if any(cells):
                        rows.append(cells)
                if rows:
                    blocks.append(("table", 0, rows))

    return blocks


def blocks_to_md(blocks):
    """Convert extracted blocks to markdown lines."""
    lines = []
    for btype, level, data in blocks:
        if btype == "heading":
            lines.append(f"{'#' * level} {data}")
            lines.append("")
        elif btype == "text":
            lines.append(data)
            lines.append("")
        elif btype == "list":
            for item in data:
                lines.append(f"- {item}")
            lines.append("")
        elif btype == "table":
            if not data:
                continue
            max_cols = max(len(r) for r in data)
            # Use first row as header
            header = data[0]
            while len(header) < max_cols:
                header.append("")
            lines.append("| " + " | ".join(h.replace("|", "/") for h in header) + " |")
            lines.append("| " + " | ".join(["---"] * max_cols) + " |")
            for row in data[1:]:
                while len(row) < max_cols:
                    row.append("")
                lines.append("| " + " | ".join(c.replace("|", "/") for c in row) + " |")
            lines.append("")
    return lines


# ====================================================================
# FILE 1: shrlo_about_and_contact.md
# ====================================================================
print("=" * 50)
print(" Crawling SHRLO About & Contact pages...")
print("=" * 50)

# --- About SHRLO ---
print("  Crawling: About SHRLO ...")
soup_about = fetch_soup("https://shrl.hkust.edu.hk/about/about-shrlo")

about_lines = [
    "# HKUST Student Housing and Residential Life Office (SHRLO) Information",
    "",
    "本文档汇总了 SHRLO 的机构介绍、联系方式和工作人员信息。",
    "",
    "---",
    "",
    "## About SHRLO (关于学生住宿及舍堂生活事务处)",
    "",
]

# Extract intro text
content_div = soup_about.select_one(".mtpc-2col-item.mtpc-2col-item--1")
if content_div:
    # Get intro paragraphs before the first h3
    for el in content_div.find_all(["p", "h3", "h4", "h5", "table"], recursive=True):
        if el.name == "h3":
            break
        if el.name == "p":
            text = el.get_text(" ", strip=True).replace("\xa0", " ")
            if text and not is_noise(text):
                about_lines.append(text)
                about_lines.append("")

    # Extract sections: Meet our Housing Officers, Hall Admission Team, etc.
    current_heading = None
    for el in content_div.find_all(["h3", "h4", "h5", "p", "table"], recursive=True):
        if el.name in ("h3", "h4", "h5"):
            text = el.get_text(" ", strip=True).replace("\xa0", " ").strip()
            if text and text != "Back":
                current_heading = text
                about_lines.append(f"### {text}")
                about_lines.append("")
        elif el.name == "p" and current_heading:
            text = el.get_text(" ", strip=True).replace("\xa0", " ")
            if text and not is_noise(text):
                about_lines.append(text)
                about_lines.append("")
        elif el.name == "table" and current_heading:
            rows = []
            for tr in el.find_all("tr"):
                cells = [c.get_text(" ", strip=True).replace("\xa0", " ")
                         for c in tr.find_all(["td", "th"])]
                if any(cells):
                    rows.append(cells)
            if rows:
                max_cols = max(len(r) for r in rows)
                # Determine header based on content
                if max_cols == 3:
                    about_lines.append("| Name | Position | Email |")
                    about_lines.append("|------|----------|-------|")
                elif max_cols == 2:
                    about_lines.append("| Name/Item | Contact/Detail |")
                    about_lines.append("|-----------|----------------|")
                else:
                    about_lines.append("| " + " | ".join([f"Col {i+1}" for i in range(max_cols)]) + " |")
                    about_lines.append("| " + " | ".join(["---"] * max_cols) + " |")
                for row in rows:
                    while len(row) < max_cols:
                        row.append("")
                    about_lines.append("| " + " | ".join(c.replace("|", "/") for c in row) + " |")
                about_lines.append("")

# --- Contact Us ---
print("  Crawling: Contact Us ...")
soup_contact = fetch_soup("https://shrl.hkust.edu.hk/about/contact-us")

about_lines.extend([
    "---",
    "",
    "## Contact Us (联系方式)",
    "",
    "If you need help, the best way to contact us is by emailing us.",
    "",
])

content_div = soup_contact.select_one(".mtpc-2col-item.mtpc-2col-item--1")
if content_div:
    current_section = None
    for el in content_div.find_all(["h3", "h4", "h5", "p", "table"], recursive=True):
        if el.name in ("h3", "h4", "h5"):
            text = el.get_text(" ", strip=True).replace("\xa0", " ").strip()
            if text and text != "Back" and text.strip():
                current_section = text
                about_lines.append(f"### {text}")
                about_lines.append("")
        elif el.name == "p":
            text = el.get_text(" ", strip=True).replace("\xa0", " ")
            if text and text != "Back" and len(text) > 2:
                about_lines.append(text)
                about_lines.append("")
        elif el.name == "table":
            rows = []
            for tr in el.find_all("tr"):
                cells = [c.get_text(" ", strip=True).replace("\xa0", " ")
                         for c in tr.find_all(["td", "th"])]
                if any(c.strip() for c in cells):
                    rows.append(cells)
            if rows:
                max_cols = max(len(r) for r in rows)
                if max_cols == 2:
                    about_lines.append("| Item | Detail |")
                    about_lines.append("|------|--------|")
                else:
                    about_lines.append("| " + " | ".join([f"Col {i+1}" for i in range(max_cols)]) + " |")
                    about_lines.append("| " + " | ".join(["---"] * max_cols) + " |")
                for row in rows:
                    while len(row) < max_cols:
                        row.append("")
                    about_lines.append("| " + " | ".join(c.replace("|", "/") for c in row) + " |")
                about_lines.append("")

    # Manually add the complete UG Hall email list to ensure completeness
    about_lines.extend([
        "### UG Hall Office Emails (本科生宿舍办公室邮箱)",
        "",
        "| Hall | Email |",
        "|------|-------|",
        "| UG Hall I | ughi@ust.hk |",
        "| UG Hall II | ughii@ust.hk |",
        "| UG Hall III | ughiii@ust.hk |",
        "| UG Hall IV | ughiv@ust.hk |",
        "| UG Hall V | ughv@ust.hk |",
        "| UG Hall VI | ughvi@ust.hk |",
        "| UG Hall VII | ughvii@ust.hk |",
        "| UG Hall VIII | ughviii@ust.hk |",
        "| UG Hall IX | ughix@ust.hk |",
        "| UG Hall X | ughx@ust.hk |",
        "| UG Hall XI & XII | ughxixii@ust.hk |",
        "| UG Hall XIII | ughxiii@ust.hk |",
        "| Jockey Club Hall | jch@ust.hk |",
        "",
    ])

write_shrlo_md("shrlo_about_and_contact.md", "\n".join(about_lines))


# ====================================================================
# FILE 2: hall_life_guide.md
#   (Safety, Living Tips, Activities, Reflection Room — combined)
# ====================================================================
print("\n" + "=" * 50)
print(" Crawling Hall Life guide pages...")
print("=" * 50)

guide_lines = [
    "# HKUST Hall Life Guide (宿舍生活指南)",
    "",
    "本文档汇总了香港科技大学宿舍生活的各项指南，包括安全守则、入住建议、",
    "活动信息和宿舍生活贴士。",
    "",
]

# --- Safety and Security ---
print("  Crawling: Safety and Security ...")
soup = fetch_soup("https://shrl.hkust.edu.hk/residential-halls/hall-life/safety-and-security")
guide_lines.extend(["---", "", "## Safety and Security (安全与保障)", ""])

content_div = soup.select_one(".mtpc-2col-item.mtpc-2col-item--1")
if content_div:
    for el in content_div.find_all(["h4", "h5", "p", "ul", "ol"], recursive=True):
        if el.name in ("h4", "h5"):
            text = el.get_text(" ", strip=True).replace("\xa0", " ").strip()
            if text and text != "Back":
                guide_lines.append(f"### {text}")
                guide_lines.append("")
        elif el.name == "p":
            text = el.get_text(" ", strip=True).replace("\xa0", " ")
            if text and text != "Back" and len(text) > 3:
                guide_lines.append(text)
                guide_lines.append("")
        elif el.name in ("ul", "ol"):
            if el.parent and el.parent.name == "li":
                continue
            for li in el.find_all("li", recursive=False):
                li_text = li.get_text(" ", strip=True).replace("\xa0", " ")
                if li_text:
                    guide_lines.append(f"- {li_text}")
            guide_lines.append("")

# --- Hall Activities ---
print("  Crawling: Hall Activities ...")
soup = fetch_soup("https://shrl.hkust.edu.hk/residential-halls/hall-life/hall-activities")
guide_lines.extend(["---", "", "## Hall Activities (宿舍活动)", ""])

content_div = soup.select_one(".mtpc-2col-item.mtpc-2col-item--1")
if content_div:
    for el in content_div.find_all(["h4", "h5", "p"], recursive=True):
        if el.name in ("h4", "h5"):
            text = el.get_text(" ", strip=True).replace("\xa0", " ").strip()
            if text and text != "Back":
                guide_lines.append(f"### {text}")
                guide_lines.append("")
        elif el.name == "p":
            text = el.get_text(" ", strip=True).replace("\xa0", " ")
            if text and text != "Back" and len(text) > 3:
                guide_lines.append(text)
                guide_lines.append("")

    # Extract activity category tabs (they are in text-area divs)
    activity_tabs = []
    for div in content_div.find_all("div", class_="field__item"):
        text = div.get_text(" ", strip=True).replace("\xa0", " ")
        if text and len(text) > 20 and "Back" not in text:
            # Check if this is an activity description
            if any(kw in text.lower() for kw in ["orientation", "program", "gathering",
                                                   "workshop", "student group", "resident"]):
                if text not in [b for _, _, b in [] ]:
                    activity_tabs.append(text)

    for tab_text in activity_tabs:
        if tab_text not in "\n".join(guide_lines):
            guide_lines.append(tab_text)
            guide_lines.append("")

# --- Living Tips ---
print("  Crawling: Living Tips ...")
soup = fetch_soup("https://shrl.hkust.edu.hk/residential-halls/hall-life/living-tips")
guide_lines.extend(["---", "", "## Living Tips (生活贴士)", ""])

content_div = soup.select_one(".mtpc-2col-item.mtpc-2col-item--1")
if content_div:
    for el in content_div.find_all(["h4", "h5", "p", "ul", "ol", "table"], recursive=True):
        if el.name in ("h4", "h5"):
            text = el.get_text(" ", strip=True).replace("\xa0", " ").strip()
            if text and text != "Back":
                guide_lines.append(f"### {text}")
                guide_lines.append("")
        elif el.name == "p":
            text = el.get_text(" ", strip=True).replace("\xa0", " ")
            if text and text != "Back" and len(text) > 3:
                guide_lines.append(text)
                guide_lines.append("")
        elif el.name in ("ul", "ol"):
            if el.parent and el.parent.name == "li":
                continue
            for li in el.find_all("li", recursive=False):
                li_text = li.get_text(" ", strip=True).replace("\xa0", " ")
                if li_text:
                    guide_lines.append(f"- {li_text}")
            guide_lines.append("")
        elif el.name == "table":
            rows = []
            for tr in el.find_all("tr"):
                cells = [c.get_text(" ", strip=True).replace("\xa0", " ")
                         for c in tr.find_all(["td", "th"])]
                if any(cells):
                    rows.append(cells)
            if rows:
                max_cols = max(len(r) for r in rows)
                lines_tbl = []
                lines_tbl.append("| " + " | ".join([f"Category {i+1}" for i in range(max_cols)]) + " |")
                lines_tbl.append("| " + " | ".join(["---"] * max_cols) + " |")
                for row in rows:
                    while len(row) < max_cols:
                        row.append("")
                    lines_tbl.append("| " + " | ".join(c.replace("|", "/") for c in row) + " |")
                guide_lines.extend(lines_tbl)
                guide_lines.append("")

# --- Reflection Room ---
print("  Crawling: Reflection Room ...")
soup = fetch_soup("https://shrl.hkust.edu.hk/hall-life-reflection-room")
guide_lines.extend(["---", "", "## Reflection Room (祈祷/冥想室)", ""])

content_div = soup.select_one(".mtpc-2col-item.mtpc-2col-item--1")
if content_div:
    for el in content_div.find_all(["h4", "h5", "p"], recursive=True):
        if el.name in ("h4", "h5"):
            text = el.get_text(" ", strip=True).replace("\xa0", " ").strip()
            if text and text != "Back":
                guide_lines.append(f"### {text}")
                guide_lines.append("")
        elif el.name == "p":
            text = el.get_text(" ", strip=True).replace("\xa0", " ")
            if text and text != "Back" and len(text) > 3:
                guide_lines.append(text)
                guide_lines.append("")

write_shrlo_md("hall_life_guide.md", "\n".join(guide_lines))


# ====================================================================
# FILE 3: hall_life_guest_pass.md
# ====================================================================
print("\n" + "=" * 50)
print(" Crawling Guest Pass Scheme...")
print("=" * 50)

print("  Crawling: Guest Pass Scheme ...")
soup = fetch_soup("https://shrl.hkust.edu.hk/residential-halls/hall-life/guest-pass-scheme")

gp_lines = [
    "# HKUST Hall Guest Pass Scheme (宿舍访客通行证计划)",
    "",
    "本文档详细记录了香港科技大学宿舍访客通行证计划的政策、规则和条款。",
    "",
]

content_div = soup.select_one(".mtpc-2col-item.mtpc-2col-item--1")
if content_div:
    current_h3 = None
    for el in content_div.find_all(["h3", "h4", "h5", "p", "ul", "ol"], recursive=True):
        if el.name == "h3":
            text = el.get_text(" ", strip=True).replace("\xa0", " ").strip()
            if text and text != "Back":
                current_h3 = text
                gp_lines.append("---")
                gp_lines.append("")
                gp_lines.append(f"## {text}")
                gp_lines.append("")
        elif el.name in ("h4", "h5"):
            text = el.get_text(" ", strip=True).replace("\xa0", " ").strip()
            if text and text != "Back":
                gp_lines.append(f"### {text}")
                gp_lines.append("")
        elif el.name == "p":
            text = el.get_text(" ", strip=True).replace("\xa0", " ")
            if text and text != "Back" and len(text) > 3:
                gp_lines.append(text)
                gp_lines.append("")
        elif el.name in ("ul", "ol"):
            if el.parent and el.parent.name == "li":
                continue
            for li in el.find_all("li", recursive=False):
                li_text = li.get_text(" ", strip=True).replace("\xa0", " ")
                if li_text:
                    # Check if it starts with a number (T&C clause)
                    if re.match(r"^\d+\.", li_text):
                        gp_lines.append(f"  {li_text}")
                    else:
                        gp_lines.append(f"- {li_text}")
            gp_lines.append("")

write_shrlo_md("hall_life_guest_pass.md", "\n".join(gp_lines))


# ====================================================================
# FILE 4: hall_life_services.md (A/C, Laundry, GGT Smartmeter)
# ====================================================================
print("\n" + "=" * 50)
print(" Crawling Hall Life Services pages...")
print("=" * 50)

svc_lines = [
    "# HKUST Hall Services (宿舍服务)",
    "",
    "本文档汇总了香港科技大学宿舍的空调服务、洗衣服务和 GGT 智能电表系统信息。",
    "",
]

# --- A/C and Laundry ---
print("  Crawling: A/C and Laundry Services ...")
soup = fetch_soup("https://shrl.hkust.edu.hk/residential-halls/hall-life/hall-ac-and-laundry-service")

svc_lines.extend(["---", "", "## Air-Conditioning and Laundry Services (空调与洗衣服务)", ""])

content_div = soup.select_one(".mtpc-2col-item.mtpc-2col-item--1")
if content_div:
    for el in content_div.find_all(["h3", "h4", "h5", "p", "ul", "ol", "table"], recursive=True):
        if el.name in ("h3", "h4"):
            text = el.get_text(" ", strip=True).replace("\xa0", " ").strip()
            if text and text != "Back":
                svc_lines.append(f"### {text}")
                svc_lines.append("")
        elif el.name == "h5":
            text = el.get_text(" ", strip=True).replace("\xa0", " ").strip()
            if text and text != "Back":
                svc_lines.append(f"**{text}**")
                svc_lines.append("")
        elif el.name == "p":
            text = el.get_text(" ", strip=True).replace("\xa0", " ")
            if text and text != "Back" and len(text) > 3:
                svc_lines.append(text)
                svc_lines.append("")
        elif el.name in ("ul", "ol"):
            if el.parent and el.parent.name == "li":
                continue
            for li in el.find_all("li", recursive=False):
                li_text = li.get_text(" ", strip=True).replace("\xa0", " ")
                if li_text:
                    svc_lines.append(f"- {li_text}")
            svc_lines.append("")
        elif el.name == "table":
            rows = []
            for tr in el.find_all("tr"):
                cells = [c.get_text(" ", strip=True).replace("\xa0", " ")
                         for c in tr.find_all(["td", "th"])]
                if any(cells):
                    rows.append(cells)
            if rows:
                max_cols = max(len(r) for r in rows)
                header = rows[0]
                while len(header) < max_cols:
                    header.append("")
                svc_lines.append("| " + " | ".join(h.replace("|", "/") for h in header) + " |")
                svc_lines.append("| " + " | ".join(["---"] * max_cols) + " |")
                for row in rows[1:]:
                    while len(row) < max_cols:
                        row.append("")
                    svc_lines.append("| " + " | ".join(c.replace("|", "/") for c in row) + " |")
                svc_lines.append("")

# Manually add laundry key info to ensure completeness
svc_lines.extend([
    "### Laundry Service Summary (洗衣服务概要)",
    "",
    "**Laundry Room Locations**:",
    "- UG Hall I and SKCC Hall: Next to the car park of SKCC Hall",
    "- UG Hall II: 2/F",
    "- UG Halls III – IX: G/F",
    "",
    "**Laundry Prices**:",
    "- Laundry with detergent provided: HK$9",
    "- Laundry with no detergent provided: HK$8",
    "- Dryer: HK$1.25 for 5 minutes",
    "",
])

# --- GGT SSC Smartmeter ---
print("  Crawling: GGT SSC Smartmeter System ...")
soup = fetch_soup("https://shrl.hkust.edu.hk/residential-halls/hall-life/ggt-ssc-smartmeter-system")

svc_lines.extend(["---", "", "## GGT SSC Smartmeter System (GGT智能电表系统)", ""])

content_div = soup.select_one(".mtpc-2col-item.mtpc-2col-item--1")
if content_div:
    for el in content_div.find_all(["h3", "h4", "h5", "p", "ul", "ol"], recursive=True):
        if el.name in ("h3", "h4"):
            text = el.get_text(" ", strip=True).replace("\xa0", " ").strip()
            if text and text != "Back":
                svc_lines.append(f"### {text}")
                svc_lines.append("")
        elif el.name == "h5":
            text = el.get_text(" ", strip=True).replace("\xa0", " ").strip()
            if text and text != "Back":
                svc_lines.append(f"**{text}**")
                svc_lines.append("")
        elif el.name == "p":
            text = el.get_text(" ", strip=True).replace("\xa0", " ")
            if text and text != "Back" and len(text) > 3:
                svc_lines.append(text)
                svc_lines.append("")
        elif el.name in ("ul", "ol"):
            if el.parent and el.parent.name == "li":
                continue
            for li in el.find_all("li", recursive=False):
                li_text = li.get_text(" ", strip=True).replace("\xa0", " ")
                if li_text:
                    svc_lines.append(f"- {li_text}")
            svc_lines.append("")

# Add known data points
svc_lines.extend([
    "### GGT Smartmeter Data Available (可查看数据)",
    "",
    "Residents can view the following information via the system:",
    "",
    "- A. Daily, weekly, and monthly real-time energy consumption for your bedroom in GGT",
    "- B. Consumption data for air-conditioning, wall sockets and lighting",
    "- C. Settle monthly utility charges invoice",
    "- D. Real-time availability of washers and dryers in all laundry rooms",
    "",
    "Access: https://w5.ab.ust.hk/njggt/app (login with HKUST Account)",
    "",
    "For enquiries, contact: ggt@ust.hk",
    "",
])

write_shrlo_md("hall_life_services.md", "\n".join(svc_lines))


# ====================================================================
# FILE 5: hall_life_checkin_checkout.md
# ====================================================================
print("\n" + "=" * 50)
print(" Crawling Check-in/out Procedures...")
print("=" * 50)

print("  Crawling: Check-in/out Procedures ...")
soup = fetch_soup("https://shrl.hkust.edu.hk/residential-halls/hall-life/check-in-out-procedures")

cio_lines = [
    "# HKUST Hall Check-in/out Procedures (宿舍入住/退宿手续)",
    "",
    "本文档汇总了香港科技大学宿舍的入住和退宿手续流程。",
    "",
]

# This page has flat HTML elements (p, h4, table, ul, ol) directly inside the
# content container — no section-item divs. We detect ## section boundaries
# by matching title patterns like "Check-In Procedures (UG)".
SECTION_TITLE_PATTERN = re.compile(
    r"^(Check-In Procedures|Check-Out Procedures|Check-in Locations|"
    r"Check-In \(UG\)|Check-Out \(UG\)|Check-In \(PG\)|Check-Out \(PG\))", re.IGNORECASE)

# Skip patterns: breadcrumb, tab labels, duplicate cell text from tables
SKIP_PATTERNS = {"Check-in/out Procedures", "Residential Halls & Hall Life / Hall Life /"}
TAB_LABELS = {"Check-In (UG)", "Check-Out (UG)", "Location (UG)",
              "Check-In (PG)", "Check-Out (PG)", "Location (PG)"}

content_div = soup.select_one(".mtpc-2col-item.mtpc-2col-item--1")
if content_div:
    seen_texts = set()  # deduplicate
    for el in content_div.find_all(["h4", "h5", "p", "ul", "ol", "table"], recursive=True):
        if el.name in ("h4", "h5"):
            text = el.get_text(" ", strip=True).replace("\xa0", " ").strip()
            if text and text != "Back" and len(text) > 2:
                cio_lines.append(f"### {text}")
                cio_lines.append("")
        elif el.name == "p":
            text = el.get_text(" ", strip=True).replace("\xa0", " ")
            if not text or text == "Back" or len(text) < 3:
                continue
            if text in SKIP_PATTERNS or text in seen_texts:
                continue
            # Check if this is a tab label list
            if all(t.strip() in TAB_LABELS for t in text.split("\n") if t.strip()):
                continue
            seen_texts.add(text)
            # Detect section title
            if SECTION_TITLE_PATTERN.match(text):
                cio_lines.extend(["---", "", f"## {text}", ""])
            else:
                cio_lines.append(text)
                cio_lines.append("")
        elif el.name in ("ul", "ol"):
            if el.parent and el.parent.name == "li":
                continue
            for li in el.find_all("li", recursive=False):
                li_text = li.get_text(" ", strip=True).replace("\xa0", " ")
                if li_text:
                    cio_lines.append(f"- {li_text}")
            cio_lines.append("")
        elif el.name == "table":
            rows = []
            for tr in el.find_all("tr"):
                cells = [c.get_text(" ", strip=True).replace("\xa0", " ")
                         for c in tr.find_all(["td", "th"])]
                if any(c.strip() for c in cells):
                    rows.append(cells)
            if rows:
                max_cols = max(len(r) for r in rows)
                header = rows[0]
                while len(header) < max_cols:
                    header.append("")
                if all(not h.strip() for h in header):
                    cio_lines.append("| Hall | Hours |")
                    cio_lines.append("|------|-------|")
                    for row in rows:
                        while len(row) < max_cols:
                            row.append("")
                        cio_lines.append("| " + " | ".join(c.replace("|", "/") for c in row) + " |")
                else:
                    cio_lines.append("| " + " | ".join(h.replace("|", "/") for h in header) + " |")
                    cio_lines.append("| " + " | ".join(["---"] * max_cols) + " |")
                    for row in rows[1:]:
                        while len(row) < max_cols:
                            row.append("")
                        cio_lines.append("| " + " | ".join(c.replace("|", "/") for c in row) + " |")
                cio_lines.append("")

write_shrlo_md("hall_life_checkin_checkout.md", "\n".join(cio_lines))


# ====================================================================
# SUMMARY
# ====================================================================
print("\n" + "=" * 60)
print(" SHRLO Info - Generated Files Summary")
print("=" * 60)

import glob
shrlo_files = sorted(glob.glob(os.path.join(SHRLO_OUTPUT_DIR, "*.md")))
for f in shrlo_files:
    name = os.path.basename(f)
    size = os.path.getsize(f)
    with open(f, "r", encoding="utf-8") as fp:
        content = fp.read()
    h2 = len(re.findall(r"^## ", content, re.MULTILINE))
    h3 = len(re.findall(r"^### ", content, re.MULTILINE))
    print(f"  {name:<40} {size/1024:.1f} KB   ## {h2:>2}   ### {h3:>2}")

print(f"\n📁 Output directory: {os.path.abspath(SHRLO_OUTPUT_DIR)}")
print("\n📋 Bailian Upload Suggestions:")
print("  - shrlo_about_and_contact.md   → 按标题切分, ## level, max 512")
print("  - hall_life_guide.md           → 按标题切分, ### level, max 512")
print("  - hall_life_guest_pass.md      → 按标题切分, ### level, max 512")
print("  - hall_life_services.md        → 按标题切分, ### level, max 512")
print("  - hall_life_checkin_checkout.md → 按标题切分, ## level, max 512")

 Crawling SHRLO About & Contact pages...
  Crawling: About SHRLO ...
  Crawling: Contact Us ...
  ✅ shrlo_about_and_contact.md  (6.2 KB, 200 lines)

 Crawling Hall Life guide pages...
  Crawling: Safety and Security ...
  Crawling: Hall Activities ...
  Crawling: Living Tips ...
  Crawling: Reflection Room ...
  ✅ hall_life_guide.md  (16.4 KB, 181 lines)

 Crawling Guest Pass Scheme...
  Crawling: Guest Pass Scheme ...
  ✅ hall_life_guest_pass.md  (9.3 KB, 108 lines)

 Crawling Hall Life Services pages...
  Crawling: A/C and Laundry Services ...
  Crawling: GGT SSC Smartmeter System ...
  ✅ hall_life_services.md  (7.0 KB, 183 lines)

 Crawling Check-in/out Procedures...
  Crawling: Check-in/out Procedures ...
  ✅ hall_life_checkin_checkout.md  (9.7 KB, 157 lines)

 SHRLO Info - Generated Files Summary
  hall_life_checkin_checkout.md            9.7 KB   ##  6   ###  5
  hall_life_guest_pass.md                  9.3 KB   ##  2   ### 14
  hall_life_guide.md                       16.4 KB   